# Tesla ML Pipeline

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

## 2. Load Data

In [ ]:
df = pd.read_csv('tesla.csv')
df.head()

In [ ]:
print(df.shape)
print(df.dtypes)

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

## 3. Preprocessing

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ','_').str.replace('-', '_')
print(df.columns.tolist())

In [ ]:
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    df['year'] = df['date'].dt.year
    df['quarter'] = df['date'].dt.quarter
    df['month'] = df['date'].dt.month

df.head()

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing after cleaning:", df.isnull().sum().sum())

## 4. EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

delivery_col = [c for c in df.columns if 'deliver' in c][0]
production_col = [c for c in df.columns if 'produc' in c][0]

axes[0].plot(df.index if 'date' not in df.columns else df['date'], df[delivery_col], marker='o', color='steelblue', label='Deliveries')
axes[0].plot(df.index if 'date' not in df.columns else df['date'], df[production_col], marker='s', color='tomato', label='Production')
axes[0].set_title('Deliveries vs Production Over Time')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

if 'year' in df.columns:
    yearly = df.groupby('year')[[delivery_col, production_col]].sum()
    yearly.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'])
    axes[1].set_title('Yearly Totals')
    axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[delivery_col], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Delivery Distribution')
axes[0].set_xlabel('Deliveries')

if 'year' in df.columns:
    df.boxplot(column=delivery_col, by='year', ax=axes[1])
    axes[1].set_title('Deliveries by Year')
    axes[1].set_xlabel('Year')
    plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
numeric_df = df[num_cols]
corr = numeric_df.corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
df['delivery_production_ratio'] = df[delivery_col] / (df[production_col] + 1)
df['rolling_3q_deliveries'] = df[delivery_col].rolling(window=3, min_periods=1).mean()
df['delivery_growth'] = df[delivery_col].pct_change().fillna(0)
df['lag_1'] = df[delivery_col].shift(1).fillna(0)
df['lag_2']= df[delivery_col].shift(2).fillna(0)

df[['delivery_production_ratio', 'rolling_3q_deliveries','delivery_growth','lag_1', 'lag_2']].head(8)

## 6. Regression Modelling

Target: predict deliveries from features.

In [ ]:
feature_cols = [c for c in df.columns if c not in [delivery_col, 'date']]
feature_cols = [c for c in feature_cols if df[c].dtype in [np.float64, np.int64, float, int]]

X = df[feature_cols].copy()
y = df[delivery_col].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

models = {
    'Linear Regression' : LinearRegression(),
    'Ridge' : Ridge(alpha=1.0),
    'Lasso' : Lasso(alpha=1.0),
    'Random Forest' : RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    Xtr = X_train_sc if name in ('Linear Regression', 'Ridge', 'Lasso') else X_train
    Xte = X_test_sc  if name in ('Linear Regression', 'Ridge', 'Lasso') else X_test
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    results[name] = {
        'MAE'  : mean_absolute_error(y_test, preds),
        'RMSE' : np.sqrt(mean_squared_error(y_test, preds)),
        'R2'   : r2_score(y_test, preds)
    }

results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
print(results_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

results_df[['MAE', 'RMSE']].plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('MAE & RMSE by Model')
axes[0].tick_params(axis='x', rotation=30)

results_df['R2'].plot(kind='bar', ax=axes[1], color='mediumseagreen')
axes[1].set_title('R² Score by Model')
axes[1].tick_params(axis='x', rotation=30)
axes[1].axhline(0, color='k', lw=0.8)

plt.tight_layout()
plt.show()

In [ ]:
best_model = RandomForestRegressor(n_estimators=100, random_state=42)
best_model.fit(X_train, y_train)
imp = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
imp.tail(15).plot(kind='barh', color='steelblue')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning

Tune Random Forest using GridSearchCV.

In [ ]:
param_grid = {
    'n_estimators' : [50, 100, 200],
    'max_depth' : [None, 5, 10],
    'min_samples_leaf': [1, 2, 5],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV R²:", round(grid_search.best_score_, 4))

In [ ]:
tuned_model = grid_search.best_estimator_
preds_tuned = tuned_model.predict(X_test)

print(f"MAE  : {mean_absolute_error(y_test, preds_tuned):.2f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, preds_tuned)):.2f}")
print(f"R²   : {r2_score(y_test, preds_tuned):.4f}")

plt.figure(figsize=(10, 5))
plt.plot(y_test.values, label='Actual',color='steelblue',  marker='o')
plt.plot(preds_tuned,label='Predicted',color='tomato',marker='s', linestyle='--')
plt.title('Tuned Model — Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Time Series Forecasting

In [ ]:
ts = df.set_index('date')[delivery_col] if 'date' in df.columns else df[delivery_col].copy()
ts.index = pd.to_datetime(ts.index) if 'date' in df.columns else ts.index

ts.plot(marker='o', color='steelblue', title='Deliveries Time Series')
plt.tight_layout()
plt.show()

In [ ]:
adf_result = adfuller(ts)
print(f"ADF Statistic : {adf_result[0]:.4f}")
print(f"p-value : {adf_result[1]:.4f}")
print(f"Stationary? : {adf_result[1] < 0.05}")

In [ ]:
ts_diff = ts.diff().dropna()
adf2 = adfuller(ts_diff)
print(f"After differencing — p-value: {adf2[1]:.4f}, Stationary? {adf2[1] < 0.05}")

In [ ]:
train_ts = ts.iloc[:-4]
test_ts  = ts.iloc[-4:]

hw_model = ExponentialSmoothing(train_ts, trend='add', seasonal=None, damped_trend=True)
hw_fit = hw_model.fit(optimized=True)
hw_fc = hw_fit.forecast(steps=4)

arima_model = ARIMA(train_ts,order=(1, 1, 1))
arima_fit = arima_model.fit()
arima_fc = arima_fit.forecast(steps=4)

plt.figure(figsize=(12, 5))
plt.plot(train_ts, label='Train', color='steelblue', marker='o')
plt.plot(test_ts, label='Actual (test)', color='black', marker='o')
plt.plot(hw_fc,label='Holt-Winters', color='tomato', linestyle='--',marker='s')
plt.plot(arima_fc,label='ARIMA(1,1,1)', color='mediumseagreen',linestyle='--', marker='^')
plt.title('Time Series Forecast — Last 4 Periods')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
for name, fc in [('Holt-Winters', hw_fc), ('ARIMA(1,1,1)', arima_fc)]:
    mae = mean_absolute_error(test_ts, fc)
    rmse = np.sqrt(mean_squared_error(test_ts, fc))
    print(f"{name:<20}  MAE={mae:.0f}  RMSE={rmse:.0f}")